In [26]:
import pandas as pd
import ast

In [7]:
sklearn_df = pd.read_csv('binary2/results_sklearn_binary2.csv')

In [23]:
def results_sklearn(df):
    # Calculate the macro average F1 score for each row
    df['average_f1'] = (df['f1_class_0'] + df['f1_class_1']) / 2
    
    # Group by vectorizer, model, and params to aggregate across seeds
    grouped = df.groupby(['vectorizer', 'model', 'params'])
    
    # Calculate the mean average F1 score for each group
    mean_f1 = grouped['average_f1'].mean().reset_index()
    
    # Find the row with the highest mean average F1 score
    best_model = mean_f1.loc[mean_f1['average_f1'].idxmax()]
    
    # Filter the original DataFrame for the best model configuration
    best_config_df = df[(df['vectorizer'] == best_model['vectorizer']) &
                        (df['model'] == best_model['model']) &
                        (df['params'] == best_model['params'])]
    
    # Calculate mean performance metrics across the three seeds
    mean_training_time = best_config_df['training_time'].mean()
    mean_prediction_time = best_config_df['prediction_time'].mean()
    mean_peak_memory_train = best_config_df['peak_memory_train'].mean()
    mean_peak_memory_prediction = best_config_df['peak_memory_prediction'].mean()
    
    # Return the results as a dictionary
    return {
        'vectorizer': best_model['vectorizer'],
        'model': best_model['model'],
        'params': best_model['params'],
        'average_f1': best_model['average_f1'],
        'mean_training_time': mean_training_time,
        'mean_prediction_time': mean_prediction_time,
        'mean_peak_memory_train': mean_peak_memory_train,
        'mean_peak_memory_prediction': mean_peak_memory_prediction
    }

In [24]:
results_sklearn(sklearn_df)

{'vectorizer': 'TfidfVectorizer',
 'model': 'LogisticRegression',
 'params': "{'C': 1, 'penalty': 'l2'}",
 'average_f1': np.float64(0.7672726760543543),
 'mean_training_time': np.float64(1.1412567000370473),
 'mean_prediction_time': np.float64(1.8686024000247319),
 'mean_peak_memory_train': np.float64(454.67578125),
 'mean_peak_memory_prediction': np.float64(445.2213541666667)}

In [47]:
bert_df = pd.read_csv('binary2/bert_binary2.csv')
bert_df

,seed,batch_size,learning_rate,train_acc,train_precisions,train_recalls,train_f1s,max_memory_usage_train,max_vram_usage_train,total_train_time,...,val_precisions,val_recalls,val_f1s,test_acc,test_precisions,test_recalls,test_f1s,max_memory_usage_test,max_vram_usage_test,total_test_time
0,2,16,0.00002,0.825088,"[0.8193964524303156, 0.8309859154929577]","[0.8339976553341149, 0.8161781946072685]","[0.826632581919591, 0.8235154956233736]",1335.171875,2528.859863,382.974231,...,"[0.8759842519685039, 0.8422939068100358]","[0.8348968105065666, 0.8818011257035647]","[0.8549471661863592, 0.8615948670944088]",0.845216,"[0.8407407407407408, 0.8498098859315589]","[0.851782363977486, 0.8386491557223265]","[0.8462255358807083, 0.8441926345609065]",1279.988281,1705.130859,9.976735
1,3,16,0.00002,0.836108,"[0.8210526315789474, 0.8526445264452645]","[0.8595545134818289, 0.8126611957796014]","[0.8398625429553265, 0.8321728691476591]",1346.320312,2534.109863,441.400862,...,"[0.8645038167938931, 0.8523985239852399]","[0.849906191369606, 0.8667917448405253]","[0.8571428571428571, 0.8595348837209302]",0.853659,"[0.840867992766727, 0.8674463937621832]","[0.8724202626641651, 0.8348968105065666]","[0.856353591160221, 0.8508604206500956]",1346.296875,1704.130859,9.566090
2,5,16,0.00002,0.832825,"[0.832046783625731, 0.8336075205640423]","[0.8339976553341149, 0.8316529894490036]","[0.8330210772833724, 0.8326291079812207]",1346.597656,2529.859863,483.820780,...,"[0.8597785977859779, 0.8721374045801527]","[0.874296435272045, 0.8574108818011257]","[0.8669767441860465, 0.8647114474929044]",0.858349,"[0.833916083916084, 0.8866396761133604]","[0.8949343339587242, 0.8217636022514071]","[0.8633484162895928, 0.8529698149951315]",1346.312500,1704.755859,10.631150


In [50]:
def results_bert(df):
    # Parse test_f1s to compute macro average F1 score for each row
    df['average_test_f1'] = df['test_f1s'].apply(
        lambda x: sum(ast.literal_eval(x)) / len(ast.literal_eval(x))
    )
    
    # Calculate mean metrics across all rows (seeds)
    mean_metrics = {
        'batch_size': df['batch_size'].iloc[0],  # Constant across rows
        'learning_rate': df['learning_rate'].iloc[0],  # Constant across rows
        'average_test_f1': df['average_test_f1'].mean(),
        'mean_train_time': df['total_train_time'].mean(),
        'mean_test_time': df['total_test_time'].mean(),
        'mean_memory_train': df['max_memory_usage_train'].mean(),
        'mean_memory_test': df['max_memory_usage_test'].mean(),
        'mean_vram_train': df['max_vram_usage_train'].mean(),
        'mean_vram_test': df['max_vram_usage_test'].mean()
    }
    
    return mean_metrics

In [51]:
results_bert(bert_df)

{'batch_size': np.int64(16),
 'learning_rate': np.float64(2e-05),
 'average_test_f1': np.float64(0.8523250689227759),
 'mean_train_time': np.float64(436.0652908999861),
 'mean_test_time': np.float64(10.057991833329046),
 'mean_memory_train': np.float64(1342.6966145833333),
 'mean_memory_test': np.float64(1324.19921875),
 'mean_vram_train': np.float64(2530.9431966145835),
 'mean_vram_test': np.float64(1704.6725260416667)}

In [52]:
lstm_df = pd.read_csv('binary2/lstm_binary2.csv')

In [54]:
def results_lstm(df):
    # Calculate mean metrics across all rows (seeds)
    mean_metrics = {
        'average_test_f1': df['test_f1'].mean(),
        'mean_train_time': df['total_time_train'].mean(),
        'mean_test_time': df['total_time_test'].mean(),
        'mean_memory_train': df['max_memory_usage_train'].mean(),
        'mean_memory_test': df['max_memory_usage_test'].mean(),
        'mean_vram_train': df['max_vram_usage_train'].mean(),
        'mean_vram_test': df['max_vram_usage_test'].mean()
    }
    
    return mean_metrics

In [55]:
results_lstm(lstm_df)

{'average_test_f1': np.float64(0.7452005141215503),
 'mean_train_time': np.float64(23.085810100000042),
 'mean_test_time': np.float64(0.7786241333621243),
 'mean_memory_train': np.float64(1244.8723958333333),
 'mean_memory_test': np.float64(1244.8658854166667),
 'mean_vram_train': np.float64(231.78580729166666),
 'mean_vram_test': np.float64(194.92447916666666)}

In [93]:
def results_llm():
    # List of files to process
    files = [
        "binary2/gemma_ZS_binary2.txt",
        "binary2/gemini_ZS_binary2.txt",
        "binary2/llama_ZS_binary2.txt",
        "binary2/openai_ZS_binary2.txt",
        "binary2/deepseekR1_ZS_binary2.txt",
    ]
    
    results = []
    
    for file_name in files:
        # Extract model name from file name
        model_name = file_name.split('_')[0].capitalize()
        
        # Initialize metric dictionary
        metrics = {
            'model': model_name,
            'f1_score': None,
            'response_time': None,
            'vram_usage': None,
            'ram_usage': None,
            'total_cost': None
        }
        
        # Read and parse the file
        try:
            with open(file_name, 'r') as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    # Split on the first colon only
                    try:
                        key, value = [x.strip() for x in line.split(':', 1)]
                    except ValueError:
                        print(f"Skipping malformed line in {file_name}: {line}")
                        continue
                    
                    # Clean value for parsing
                    if key == 'Total cost':
                        value = value.replace('USD', '').strip()
                    
                    # Try converting value to float
                    try:
                        value = float(value)
                    except ValueError:
                        print(f"Skipping non-numeric value in {file_name}: {key} = {value}")
                        continue
                    
                    # Assign to appropriate metric
                    if key == 'F1 score':
                        metrics['f1_score'] = value
                    elif key == 'Average response time':
                        metrics['response_time'] = value
                    elif key == 'Average VRAM usage':
                        metrics['vram_usage'] = value
                    elif key == 'Average RAM usage':
                        metrics['ram_usage'] = value
                    elif key == 'Total cost':
                        metrics['total_cost'] = value
                        print(f"Parsed cost for {model_name}: {value}")  # Debug
                
        except FileNotFoundError:
            print(f"Warning: File {file_name} not found.")
            continue
        
        results.append(metrics)
    
    return results

In [94]:
llm = results_llm()
for res in llm:
    print(f"Model: {res['model']}, F1 Score: {res['f1_score']}, "
          f"Response Time: {res['response_time']}, "
          f"VRAM Usage: {res['vram_usage']}, RAM Usage: {res['ram_usage']}")

Parsed cost for Binary2/gemini: 0.011469300000000017
Parsed cost for Binary2/openai: 0.01921649999999998
Model: Binary2/gemma, F1 Score: 0.849327713405063, Response Time: 2.32628433945926, VRAM Usage: 5242.090994371482, RAM Usage: 102.9576835131332
Model: Binary2/gemini, F1 Score: 0.9286750619788147, Response Time: 0.9638142916767056, VRAM Usage: None, RAM Usage: None
Model: Binary2/llama, F1 Score: 0.8699828277862095, Response Time: 2.269008244179874, VRAM Usage: 4121.598499061914, RAM Usage: 91.32670321294559
Model: Binary2/openai, F1 Score: 0.8952960964216534, Response Time: 0.7937980326657894, VRAM Usage: None, RAM Usage: None
Model: Binary2/deepseekr1, F1 Score: 0.5618647812638091, Response Time: 5.459199921504871, VRAM Usage: 2432.5135135135133, RAM Usage: 90.71267947635135
